# 02 — COCO-standard 10% split + unlabeled gallery pool (v2)

**Project:** RAOD++ · **Phase:** F4 · **Protocol:** v1.0 (frozen)

Prepares the secondary benchmark behind contribution **K3**: a 10% labeled subset for
supervision, with the remaining 90% kept as a genuinely *unlabeled* retrieval gallery.

Data is already on disk (Kaggle mirror `awsaf49/coco-2017-dataset`, extracted to
`dataset/coco/coco2017/`) because cocodataset.org is blocked on this network. This
notebook therefore does **not** download anything.

What it does:
1. verifies the extracted data,
2. converts instance annotations to YOLO labels,
3. builds and freezes the 10% labeled split (fold-1),
4. **quarantines the ground truth of the remaining 90%**,
5. runs a 4-point GT-isolation audit,
6. writes the data YAML, the gallery manifest and the data card.

**Runtime:** 10–20 min, CPU only. Safe to run while E0 trains on the GPU.


## [Cell 3] Config and integrity check
`OFFICIAL_SPLIT_FILE`: SSOD papers use released fixed split lists. If you obtain one,
put its path here and it is used verbatim. Otherwise a seeded random 10% sample is
generated and that fact is recorded in the data card and stated in the manuscript.

In [ ]:
# ===== Cell 3 =====
from pathlib import Path

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "DATASET_NAME": "COCO-2017 (10% labeled protocol, fold-1)",
    "SOURCE": "Kaggle mirror awsaf49/coco-2017-dataset",
    "LABELED_FRACTION": 0.10,
    "FOLD": 1,
    "SEED": 42,
    "OFFICIAL_SPLIT_FILE": None,
    "EXPECTED": {"train2017": 118287, "val2017": 5000},
}

ROOT = CONFIG["PROJECT_ROOT"]
DIRS = {k: ROOT / k for k in ["dataset","model","output","fig","tex","notebooks"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

COCO = DIRS["dataset"] / "coco" / "coco2017"
ANN_DIR = COCO / "annotations"
assert (ANN_DIR/"instances_train2017.json").exists(), f"annotations missing under {ANN_DIR}"

ok = True
for split, exp in CONFIG["EXPECTED"].items():
    n = len(list((COCO/split).glob("*.jpg")))
    ok &= (n == exp)
    print(f"{split}: found={n} expected={exp} [{'OK' if n==exp else 'MISMATCH'}]")
assert ok, "Image counts do not match the expected COCO 2017 sizes"

print("COCO root  :", COCO)
print("split mode :", "OFFICIAL FILE" if CONFIG["OFFICIAL_SPLIT_FILE"]
      else f"SEEDED RANDOM (seed={CONFIG['SEED']}+fold{CONFIG['FOLD']})")


## [Cell 5] Convert instance annotations to YOLO labels
Labels go to a staging directory first. Nothing is exposed to training until the
labeled/unlabeled separation is made in the next cells. Takes a few minutes.

In [ ]:
# ===== Cell 5 =====
import shutil
from ultralytics.data.converter import convert_coco

STAGE = COCO / "labels_staging"
if STAGE.exists() and not any(STAGE.rglob("*.txt")):
    shutil.rmtree(STAGE)                       # clear an earlier empty attempt
if not STAGE.exists():
    convert_coco(labels_dir=str(ANN_DIR), save_dir=str(STAGE),
                 use_segments=False, use_keypoints=False, cls91to80=True)

label_dirs = {d: len(list(d.glob("*.txt")))
              for d in STAGE.rglob("*") if d.is_dir() and any(d.glob("*.txt"))}
assert label_dirs, f"Conversion produced no .txt files under {STAGE}"

def pick(tag):
    hits = [d for d in label_dirs if tag in str(d).lower()]
    assert hits, f"No label dir matching '{tag}'. Found: {[str(d) for d in label_dirs]}"
    return max(hits, key=lambda d: label_dirs[d])

stage_train, stage_val = pick("train2017"), pick("val2017")
for d, n in sorted(label_dirs.items(), key=lambda kv: -kv[1]):
    print(f"  {d.relative_to(STAGE)}: {n} txt")
print(f"\nstage_train={label_dirs[stage_train]}  stage_val={label_dirs[stage_val]}")


## [Cell 7] Build and freeze the 10% labeled split
Images without annotations are excluded from the labeled candidate pool (they carry no
supervision) but remain eligible for the unlabeled gallery.

In [ ]:
# ===== Cell 7 =====
import random, hashlib, json

all_imgs = sorted(p.name for p in (COCO/"train2017").glob("*.jpg"))
annotated = {p.stem for p in stage_train.glob("*.txt") if p.stat().st_size > 0}
candidates = [n for n in all_imgs if Path(n).stem in annotated]
print(f"train2017: total={len(all_imgs)}  annotated={len(candidates)}")

if CONFIG["OFFICIAL_SPLIT_FILE"]:
    labeled = sorted(Path(CONFIG["OFFICIAL_SPLIT_FILE"]).read_text().split())
    split_mode = f"official_file:{Path(CONFIG['OFFICIAL_SPLIT_FILE']).name}"
else:
    rng = random.Random(CONFIG["SEED"] + CONFIG["FOLD"])
    k = int(round(CONFIG["LABELED_FRACTION"] * len(candidates)))
    labeled = sorted(rng.sample(candidates, k))
    split_mode = f"seeded_random(seed={CONFIG['SEED']}+fold{CONFIG['FOLD']})"

labeled_set = set(labeled)
unlabeled = [n for n in all_imgs if n not in labeled_set]

hashes = {"split_mode": split_mode}
for name, lst in [("labeled", labeled), ("unlabeled", unlabeled)]:
    txt = "\n".join(lst)
    (DIRS["output"]/f"coco10_{name}_files.txt").write_text(txt)
    hashes[name] = {"n": len(lst), "sha256": hashlib.sha256(txt.encode()).hexdigest()}
    print(f"{name}: n={len(lst)}  sha256={hashes[name]['sha256'][:16]}...")

json.dump(hashes, open(DIRS["output"]/"coco10_split_hashes.json","w"), indent=2)
print("Split FROZEN. mode =", split_mode)


## [Cell 9] Materialize the split and quarantine the unlabeled ground truth

```
coco2017/images/train_labeled/   + labels/train_labeled/   supervision (10%)
coco2017/images/train_unlabeled/                           gallery pool, NO labels dir
coco2017/images/val2017/         + labels/val2017/         evaluation only
coco2017/_quarantine/unlabeled_labels/                     the 90% GT, never referenced
```

Symlinks are used, so no image is duplicated on disk.

In [ ]:
# ===== Cell 9 =====
import os

IMG, LBL = COCO/"images", COCO/"labels"
QUAR = COCO/"_quarantine"/"unlabeled_labels"
for d in [IMG/"train_labeled", IMG/"train_unlabeled", IMG/"val2017",
          LBL/"train_labeled", LBL/"val2017", QUAR]:
    d.mkdir(parents=True, exist_ok=True)

def link(src, dst):
    if not dst.exists():
        try: os.symlink(src, dst)
        except FileExistsError: pass

for n in labeled:
    link(COCO/"train2017"/n, IMG/"train_labeled"/n)
    s = stage_train/(Path(n).stem + ".txt")
    if s.exists(): link(s, LBL/"train_labeled"/s.name)

for n in unlabeled:
    link(COCO/"train2017"/n, IMG/"train_unlabeled"/n)
    s = stage_train/(Path(n).stem + ".txt")
    if s.exists(): link(s, QUAR/s.name)          # quarantined, off every data path

for p in (COCO/"val2017").glob("*.jpg"): link(p, IMG/"val2017"/p.name)
for p in stage_val.glob("*.txt"):        link(p, LBL/"val2017"/p.name)

for d in [IMG/"train_labeled", LBL/"train_labeled", IMG/"train_unlabeled",
          IMG/"val2017", LBL/"val2017", QUAR]:
    print(f"{str(d).replace(str(COCO),'coco2017'):48s} {len(list(d.iterdir())):7d}")


## [Cell 11] GT-isolation audit
Four hard checks. Contribution K3 rests entirely on these; any failure aborts.

In [ ]:
# ===== Cell 11 =====
# 1) no label directory next to the unlabeled images
assert not (LBL/"train_unlabeled").exists(), "FAIL: labels dir exists for unlabeled pool"

# 2) ultralytics resolves labels by swapping /images/ -> /labels/; that path must be empty
implied = Path(str(IMG/"train_unlabeled").replace("/images/","/labels/"))
assert not implied.exists() or not any(implied.iterdir()), f"FAIL: {implied} reachable"

# 3) labeled and unlabeled are disjoint and jointly cover train2017
L = {p.name for p in (IMG/"train_labeled").iterdir()}
U = {p.name for p in (IMG/"train_unlabeled").iterdir()}
assert not (L & U), f"FAIL: {len(L & U)} images in both sets"
assert len(L | U) == len(all_imgs), "FAIL: coverage mismatch"

# 4) no training image leaks into evaluation
V = {p.name for p in (IMG/"val2017").iterdir()}
assert not ((L | U) & V), "FAIL: train/val overlap"

print(f"labeled={len(L)}  unlabeled={len(U)}  val={len(V)}")
print(f"labeled fraction = {len(L)/len(all_imgs):.4f}")
print("GT-isolation audit: PASS (4/4)")


## [Cell 13] Data YAML and gallery manifest
The YAML deliberately lists only the labeled split. The gallery pool is referenced
through a separate manifest so it can never be mistaken for supervised data.

In [ ]:
# ===== Cell 13 =====
import yaml as _yaml, ultralytics

coco_names = _yaml.safe_load(
    open(Path(ultralytics.__file__).parent/"cfg"/"datasets"/"coco.yaml"))["names"]

yaml_path = DIRS["dataset"]/"coco10_raodpp.yaml"
yaml_path.write_text(
    "# RAOD++ COCO 10% protocol (fold-1) — generated by 02_data_coco10.ipynb\n"
    "# train_unlabeled is deliberately NOT listed: it is an unlabeled gallery pool.\n"
    f"path: {COCO}\ntrain: images/train_labeled\nval: images/val2017\nnames:\n"
    + "\n".join(f"  {i}: {n}" for i, n in coco_names.items()) + "\n")

manifest = DIRS["output"]/"coco10_gallery_manifest.txt"
manifest.write_text("\n".join(str(IMG/"train_unlabeled"/n) for n in unlabeled))

print(yaml_path.read_text()[:300], "...\n")
print("gallery manifest:", manifest, "|", len(unlabeled), "images")


## [Cell 15] Data card

In [ ]:
# ===== Cell 15 =====
import pandas as pd

n_lbl_boxes = sum(1 for p in (LBL/"train_labeled").iterdir() for _ in open(p))
n_val_boxes = sum(1 for p in (LBL/"val2017").iterdir() for _ in open(p))

card = pd.DataFrame([
    ["Dataset", CONFIG["DATASET_NAME"]],
    ["Role in study", "Secondary benchmark (contribution K3)"],
    ["Source", CONFIG["SOURCE"]],
    ["Labeled train images", len(L)],
    ["Unlabeled gallery images", len(U)],
    ["Eval images (val2017)", len(V)],
    ["Classes", len(coco_names)],
    ["Labeled instances", n_lbl_boxes],
    ["Eval instances", n_val_boxes],
    ["Labeled fraction", f"{len(L)/len(all_imgs):.4f}"],
    ["Split mode", split_mode],
    ["GT isolation", "quarantined; 4/4 audit passed"],
    ["Image size (training)", "640 (protocol v1.0)"],
    ["License / access", "COCO 2017, annotations CC BY 4.0"],
    ["Split hash (labeled)", hashes["labeled"]["sha256"][:32]+"..."],
], columns=["field","value"])
card.to_csv(DIRS["output"]/"datacard_coco10.csv", index=False)
card


**Next:** `03_data_exdark.ipynb` closes F4 (ExDark download, conversion, the
3000/1800/2563 split used in the low-light detection literature, data card).

Report back: the two label counts, `PASS (4/4)`, and the labeled fraction.